In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "72e7743b",
   "metadata": {},
   "outputs": [],
   "source": [
    "# -*- coding: utf-8 -*-\n",
    "\"\"\"\n",
    "PCA vs t-SNE — Comparación 2D y 3D sobre Digits (scikit-learn)\n",
    "==============================================================\n",
    "\n",
    "Este script:\n",
    "- Carga y estandariza el dataset Digits.\n",
    "- Aplica PCA (2D, 3D) y t-SNE (2D, 3D).\n",
    "- Genera 4 gráficas (PNG) y calcula métricas: varianza (PCA), trustworthiness y KNN (5-fold).\n",
    "- Imprime una tabla resumen e interpretación.\n",
    "\n",
    "Compatibilidad:\n",
    "- t-SNE SIN 'n_iter' y con 'learning_rate' numérico (evita errores en scikit-learn antiguos).\n",
    "\"\"\"\n",
    "\n",
    "# ---------- 1) Imports ----------\n",
    "import numpy as np                       # Cálculo numérico\n",
    "import pandas as pd                      # Tabla de resultados\n",
    "import matplotlib.pyplot as plt          # Gráficas (no estilos extra)\n",
    "from mpl_toolkits.mplot3d import Axes3D  # Habilita proyección 3D en Matplotlib\n",
    "import time                              # Medir tiempos\n",
    "\n",
    "from sklearn.datasets import load_digits                 # Dataset Digits (8x8=64 features)\n",
    "from sklearn.preprocessing import StandardScaler         # Estandarización (media 0, var 1)\n",
    "from sklearn.decomposition import PCA                    # PCA (lineal)\n",
    "from sklearn.manifold import TSNE, trustworthiness       # t-SNE y métrica de vecindad\n",
    "from sklearn.model_selection import StratifiedKFold, cross_val_score  # Validación cruzada estratificada\n",
    "from sklearn.neighbors import KNeighborsClassifier       # Clasificador KNN (k=5)\n",
    "\n",
    "# ---------- 2) Parámetros ----------\n",
    "SUBMUESTREO = 600               # Usar 600 muestras para acelerar t-SNE; pon None para usar todas\n",
    "RANDOM_STATE = 42               # Reproducibilidad\n",
    "TSNE_PERPLEXITY = 20            # Tamaño de vecindario efectivo (5–50)\n",
    "TSNE_LEARNING_RATE = 200        # Valor numérico (evita 'auto' por compatibilidad)\n",
    "TSNE_EARLY_EXAGGERATION = 8     # Acelera separación inicial (típico 8–12)\n",
    "\n",
    "# ---------- 3) Cargar datos + submuestrear + escalar ----------\n",
    "digits = load_digits()          # Carga (n≈1797, d=64)\n",
    "X_full, y_full = digits.data, digits.target  # X: features (64), y: dígitos 0..9\n",
    "\n",
    "if SUBMUESTREO is not None and SUBMUESTREO < len(X_full):\n",
    "    rng = np.random.RandomState(RANDOM_STATE)         # RNG reproducible\n",
    "    idx = rng.choice(len(X_full), size=SUBMUESTREO, replace=False)  # Índices aleatorios sin reemplazo\n",
    "    X, y = X_full[idx], y_full[idx]                   # Subconjunto (p.ej., n=600)\n",
    "else:\n",
    "    X, y = X_full, y_full                             # Usa todo el dataset\n",
    "\n",
    "scaler = StandardScaler()                             # Crea escalador estándar\n",
    "X_scaled = scaler.fit_transform(X)                    # Ajusta en X y transforma → media 0, var 1\n",
    "\n",
    "# ---------- 4) PCA 2D ----------\n",
    "t0 = time.perf_counter()                              # Cronómetro inicio\n",
    "pca2 = PCA(n_components=2, random_state=RANDOM_STATE) # PCA a 2 componentes (PC1, PC2)\n",
    "X_pca2 = pca2.fit_transform(X_scaled)                 # Ajusta PCA y proyecta a 2D\n",
    "pca2_time = time.perf_counter() - t0                  # Tiempo de PCA 2D\n",
    "pca2_var = pca2.explained_variance_ratio_.sum()       # Varianza acumulada en 2D\n",
    "\n",
    "# ----- Gráfico PCA 2D -----\n",
    "plt.figure()                                          # Nueva figura\n",
    "plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=y, s=12, alpha=0.85)  # Colorea por clase\n",
    "plt.title(f\"PCA 2D — Varianza acumulada: {pca2_var:.3f}\")       # Título con varianza\n",
    "plt.xlabel(\"PC1\"); plt.ylabel(\"PC2\")                  # Ejes\n",
    "plt.tight_layout()                                    # Ajuste de márgenes\n",
    "plt.savefig(\"pca_2d.png\", dpi=150)                    # Guarda imagen\n",
    "# plt.show()                                          # Descomenta si quieres ver en pantalla\n",
    "\n",
    "# ---------- 5) PCA 3D ----------\n",
    "t0 = time.perf_counter()                              # Cronómetro inicio\n",
    "pca3 = PCA(n_components=3, random_state=RANDOM_STATE) # PCA a 3 componentes (PC1, PC2, PC3)\n",
    "X_pca3 = pca3.fit_transform(X_scaled)                 # Proyección a 3D\n",
    "pca3_time = time.perf_counter() - t0                  # Tiempo de PCA 3D\n",
    "pca3_var = pca3.explained_variance_ratio_.sum()       # Varianza acumulada en 3D\n",
    "\n",
    "# ----- Gráfico PCA 3D -----\n",
    "fig = plt.figure()                                    # Nueva figura\n",
    "ax = fig.add_subplot(111, projection='3d')            # Eje 3D\n",
    "ax.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2], c=y, s=12, alpha=0.85)  # Scatter 3D\n",
    "ax.set_title(f\"PCA 3D — Varianza acumulada: {pca3_var:.3f}\")    # Título con varianza\n",
    "ax.set_xlabel(\"PC1\"); ax.set_ylabel(\"PC2\"); ax.set_zlabel(\"PC3\")# Etiquetas ejes\n",
    "plt.tight_layout()                                    # Ajuste de márgenes\n",
    "plt.savefig(\"pca_3d.png\", dpi=150)                    # Guarda imagen\n",
    "# plt.show()\n",
    "\n",
    "# ---------- 6) t-SNE 2D (compatibilidad: sin n_iter, LR numérico) ----------\n",
    "t0 = time.perf_counter()                              # Cronómetro inicio\n",
    "tsne2 = TSNE(\n",
    "    n_components=2,                                   # Embebido a 2D\n",
    "    perplexity=TSNE_PERPLEXITY,                       # Vecindario efectivo\n",
    "    learning_rate=TSNE_LEARNING_RATE,                 # Tasa de aprendizaje (numérico)\n",
    "    init=\"pca\",                                       # Inicialización PCA (converge más estable)\n",
    "    early_exaggeration=TSNE_EARLY_EXAGGERATION,       # Exageración inicial\n",
    "    random_state=RANDOM_STATE,                        # Reproducible\n",
    ")\n",
    "X_tsne2 = tsne2.fit_transform(X_scaled)               # Ajusta y transforma\n",
    "tsne2_time = time.perf_counter() - t0                 # Tiempo t-SNE 2D\n",
    "\n",
    "# ----- Gráfico t-SNE 2D -----\n",
    "plt.figure()\n",
    "plt.scatter(X_tsne2[:, 0], X_tsne2[:, 1], c=y, s=12, alpha=0.85)\n",
    "plt.title(\"t-SNE 2D\")\n",
    "plt.xlabel(\"Dim 1\"); plt.ylabel(\"Dim 2\")\n",
    "plt.tight_layout()\n",
    "plt.savefig(\"tsne_2d.png\", dpi=150)\n",
    "# plt.show()\n",
    "\n",
    "# ---------- 7) t-SNE 3D ----------\n",
    "t0 = time.perf_counter()                              # Cronómetro inicio\n",
    "tsne3 = TSNE(\n",
    "    n_components=3,                                   # Embebido a 3D\n",
    "    perplexity=TSNE_PERPLEXITY,                       # Vecindario efectivo\n",
    "    learning_rate=TSNE_LEARNING_RATE,                 # Tasa de aprendizaje\n",
    "    init=\"pca\",                                       # Inicialización\n",
    "    early_exaggeration=TSNE_EARLY_EXAGGERATION,       # Exageración inicial\n",
    "    random_state=RANDOM_STATE,                        # Reproducible\n",
    ")\n",
    "X_tsne3 = tsne3.fit_transform(X_scaled)               # Ajusta y transforma\n",
    "tsne3_time = time.perf_counter() - t0                 # Tiempo t-SNE 3D\n",
    "\n",
    "# ----- Gráfico t-SNE 3D -----\n",
    "fig = plt.figure()\n",
    "ax = fig.add_subplot(111, projection='3d')\n",
    "ax.scatter(X_tsne3[:, 0], X_tsne3[:, 1], X_tsne3[:, 2], c=y, s=12, alpha=0.85)\n",
    "ax.set_title(\"t-SNE 3D\")\n",
    "ax.set_xlabel(\"Dim 1\"); ax.set_ylabel(\"Dim 2\"); ax.set_zlabel(\"Dim 3\")\n",
    "plt.tight_layout()\n",
    "plt.savefig(\"tsne_3d.png\", dpi=150)\n",
    "# plt.show()\n",
    "\n",
    "# ---------- 8) Métricas: trustworthiness y KNN ----------\n",
    "tw_pca2  = trustworthiness(X_scaled, X_pca2, n_neighbors=5)     # PCA 2D\n",
    "tw_pca3  = trustworthiness(X_scaled, X_pca3, n_neighbors=5)     # PCA 3D\n",
    "tw_tsne2 = trustworthiness(X_scaled, X_tsne2, n_neighbors=5)    # t-SNE 2D\n",
    "tw_tsne3 = trustworthiness(X_scaled, X_tsne3, n_neighbors=5)    # t-SNE 3D\n",
    "\n",
    "cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)  # Particiones estratificadas\n",
    "knn = KNeighborsClassifier(n_neighbors=5)                                   # KNN con k=5\n",
    "\n",
    "acc_orig  = cross_val_score(knn, X_scaled, y, cv=cv, scoring=\"accuracy\")    # 64D original\n",
    "acc_pca2  = cross_val_score(knn, X_pca2,  y, cv=cv, scoring=\"accuracy\")     # PCA 2D\n",
    "acc_pca3  = cross_val_score(knn, X_pca3,  y, cv=cv, scoring=\"accuracy\")     # PCA 3D\n",
    "acc_tsne2 = cross_val_score(knn, X_tsne2, y, cv=cv, scoring=\"accuracy\")     # t-SNE 2D (ilustrativo)\n",
    "acc_tsne3 = cross_val_score(knn, X_tsne3, y, cv=cv, scoring=\"accuracy\")     # t-SNE 3D (ilustrativo)\n",
    "\n",
    "# ---------- 9) Tabla resumen ----------\n",
    "summary = pd.DataFrame({\n",
    "    \"Método\": [\n",
    "        f\"Original (64D, n={X_scaled.shape[0]})\",  # Sin reducción\n",
    "        \"PCA 2D\", \"PCA 3D\",                        # PCA 2/3D\n",
    "        \"t-SNE 2D\", \"t-SNE 3D\"                     # t-SNE 2/3D\n",
    "    ],\n",
    "    \"Dimensiones\": [X_scaled.shape[1], 2, 3, 2, 3],            # Número de dimensiones del espacio\n",
    "    \"Tiempo_fit (s)\": [np.nan, pca2_time, pca3_time, tsne2_time, tsne3_time],  # Tiempos\n",
    "    \"Varianza acumulada (PCA)\": [np.nan, pca2_var, pca3_var, np.nan, np.nan],  # Solo PCA\n",
    "    \"Trustworthiness (k=5)\": [np.nan, tw_pca2, tw_pca3, tw_tsne2, tw_tsne3],   # Preservación local\n",
    "    \"KNN Acc media (5-fold)\": [\n",
    "        acc_orig.mean(), acc_pca2.mean(), acc_pca3.mean(), acc_tsne2.mean(), acc_tsne3.mean()\n",
    "    ],\n",
    "    \"KNN Acc std (5-fold)\": [\n",
    "        acc_orig.std(), acc_pca2.std(), acc_pca3.std(), acc_tsne2.std(), acc_tsne3.std()\n",
    "    ],\n",
    "})\n",
    "\n",
    "# ---------- 10) Impresión + orientación ----------\n",
    "pd.set_option(\"display.max_columns\", None)           # Muestra todas las columnas al imprimir\n",
    "print(\"\\n=== COMPARACIÓN PCA vs t-SNE — 2D y 3D ===\")\n",
    "print(summary.to_string(index=False))                # Muestra tabla completa\n",
    "\n",
    "print(\"\\nInterpretación rápida:\")\n",
    "print(\"- PCA captura varianza global; 3D suele mejorar frente a 2D (ver 'Varianza acumulada').\")\n",
    "print(\"- t-SNE preserva vecindarios; normalmente alcanza trustworthiness alto, pero tarda más.\")\n",
    "print(\"- KNN: en 64D (original) suele ser fuerte; PCA-3D > PCA-2D; t-SNE 2D/3D es para visualizar, no para producción.\")\n",
    "\n",
    "print(\"\\nImágenes guardadas:\")\n",
    "print(\" - pca_2d.png\")\n",
    "print(\" - pca_3d.png\")\n",
    "print(\" - tsne_2d.png\")\n",
    "print(\" - tsne_3d.png\")"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}